# Pre-entrega 6: demostración de delegación multi-agente

Este notebook ejecuta la misma topología jerárquica del repositorio: **Supervisor → especialistas → Supervisor**. No contiene credenciales ni resultados pregrabados.

## 1. Instalación

En Google Colab, descomentá y ejecutá la siguiente celda después de clonar el repositorio.

In [ ]:
# !git clone https://github.com/rguima-ai/langgraph-multi-agent-orchestrator.git
# %cd langgraph-multi-agent-orchestrator
# !pip install -e .

## 2. API key

En Colab conviene guardar `OPENAI_API_KEY` dentro de **Secrets** y no escribirla en una celda.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass  # En local se usa el archivo .env

## 3. Construcción del equipo

Cada especialista recibe el mismo modelo, pero tiene un prompt, herramientas y campos de escritura diferentes.

In [ ]:
from langchain_openai import ChatOpenAI

from multi_agent_orchestrator.agents import AnalystAgent, ResearchAgent, SynthesisAgent
from multi_agent_orchestrator.graph import TeamNodes, build_graph
from multi_agent_orchestrator.supervisor import SupervisorNode

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
nodes = TeamNodes(
    supervisor=SupervisorNode(model),
    researcher=ResearchAgent(model),
    analyst=AnalystAgent(model),
    synthesizer=SynthesisAgent(model),
)
graph = build_graph(nodes)

## 4. Estado inicial

El pedido obliga a usar dos dominios: investigación y cálculo, antes de sintetizar.

In [ ]:
from langchain_core.messages import HumanMessage

request = (
    "Investigá el crecimiento del mercado de energía solar entre 2023 y 2024, "
    "calculá la variación porcentual y redactá una recomendación ejecutiva."
)
initial_state = {
    "messages": [HumanMessage(content=request)],
    "request": request,
    "research_output": None,
    "analysis_output": None,
    "final_report": None,
    "task_completed": False,
    "step_count": 0,
    "max_steps": 8,
    "contributions": [],
    "errors": [],
}

## 5. Streaming de la delegación

`astream(..., stream_mode="updates")` permite observar qué nodo actualiza el estado en cada paso.

In [ ]:
final_report = None
async for update in graph.astream(
    initial_state,
    config={"recursion_limit": 24},
    stream_mode="updates",
):
    for node, values in update.items():
        if node == "supervisor":
            print(f"Supervisor → {values.get('next_agent')}: {values.get('supervisor_reason')}")
        else:
            print(f"{node}: completado")
        if values.get("final_report"):
            final_report = values["final_report"]

## 6. Resultado final

In [ ]:
from pprint import pprint

pprint(final_report)